# <font color="#003660">Applied Machine Learning for Text Analysis (M.184.5331)</font>


# <font color="#003660">Session 1: Introduction to Natural Language Processing</font>

# <font color="#003660">Notebook 1a: Document Classification</font>

<center><br><img width=256 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/dag.png"/><br></center>

<p>
<center>
<div>
    <font color="#085986"><b>By the end of this lesson, you will be able to...</b><br><br>
        ... transform raw text into a term-document matrix, <br>
        ... train a binary classifier on the term-document matrix, and <br> ... and compete in a <a href="https://www.kaggle.com/t/208059e8f1434629a60386479a523af0">Kaggle competition</a>.
    </font>
</div>
</center>
</p>

# Import packages

As always, we first need to load a number of required Python packages:
- `pandas` provides high-performance, easy-to-use data structures and data analysis tools.
- `spacy` offers industrial-strength natural language processing.
- `sklearn` is the de-facto standard machine learning package in Python.

In [ ]:
#%pip install -q spacy-lookups-data

In [ ]:
import pandas as pd
import spacy
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn import metrics

# Load documents

Load 100,000 labeled wine reviews (Source: https://www.kaggle.com/datasets/zynicide/wine-reviews) from a csv file.

In [ ]:
corpus = pd.read_csv("https://raw.githubusercontent.com/olivermueller/amlta-2026/main/Session_01/wine_reviews_train.csv")

In [ ]:
corpus.head()

In [ ]:
corpus.shape

# Preprocess documents

Create response variable.

In [ ]:
corpus["verygood"] = 0
corpus.loc[corpus['points'] > 90, 'verygood'] = 1

Split the labeled data into a training and a validation set. The test set for the Kaggle competition comes in a separate file without labels (we load it at the end of this notebook).

In [ ]:
training = corpus.iloc[0:80000,]
validation = corpus.iloc[80000:,]

In [ ]:
print(training.shape)
print(validation.shape)

Perform standard NLP preprocessing steps on the training set using spaCy. spaCy is an open-source library for Natural Language Processing (NLP) in Python. It helps you build NLP applications that process and understand large volumes of unstructured text. One of the main features of spaCy are linguistic annotations that give you insights into a text’s grammatical structure (e.g., word order, types of words, parts of speech, grammatical roles and relations).

At the center of spaCy is the processing pipeline, an object which is usually called `nlp`. The pipeline is build on top of a language-specific machine learning model and a set of handcrafted rules.

The pipeline contains different components, each specialized for a specific NLP task.

[More...](https://spacy.io/usage/spacy-101#whats-spacy)

<center><br><img src="https://spacy.io/images/pipeline.svg"/><br></center>

spaCy offers pre-trained statistical pipelines (e.g., `en_core_web_sm`) that predict part-of-speech tags, dependencies, and named entities with a neural network. For our preprocessing, we only need tokens, stop words, and lemmas. We therefore start from a blank English pipeline (which only contains the tokenizer) and add a *lookup lemmatizer*, which simply looks up the base form of each word in a dictionary. This is roughly 25 times faster than running the statistical pipeline, at the price of occasional mistakes, because the lookup lemmatizer ignores context (e.g., it cannot tell whether "bit" is a noun or the past tense of "bite").

In [ ]:
nlp = spacy.blank("en")
nlp.add_pipe("lemmatizer", config={"mode": "lookup"})
nlp.initialize()

def spacy_prep_df(df, text_col="description", batch_size=1000):
    texts = (str(x).lower() if pd.notna(x) and x else "" for x in df[text_col].values)  # the lookup lemmatizer is case-sensitive
    results = []
    for doc in nlp.pipe(texts, batch_size=batch_size):
        toks = [t.lemma_ for t in doc if t.is_alpha and not t.is_stop]
        results.append(" ".join(toks))
    out = df.copy()
    out[text_col + "_prep"] = results
    return out


In [ ]:
training = spacy_prep_df(training)

Display the first couple of lines of the preprocessed descriptions.

In [ ]:
training["description_prep"].head()

## Excursus: Which lemmatizer should we use?

*Lemmatization* reduces each word to its dictionary base form (its *lemma*), e.g., "cherries" → "cherry", "was" → "be". This way, different forms of the same word end up in the same column of the term-document matrix. spaCy offers different ways to do this:

| Option | How it works | Pros | Cons |
|---|---|---|---|
| **Lookup lemmatizer** (what we use above) | Looks up each word in a big table of word → lemma pairs (package `spacy-lookups-data`) | Very fast, no model needed | Ignores context; case-sensitive (that's why we lowercase the texts first); unknown words stay unchanged |
| **Rule-based lemmatizer** in a statistical pipeline (`en_core_web_sm`, `_md`, `_lg`) | A small neural network first predicts the part of speech (POS) of each word, then POS-specific rules and exception lists determine the lemma | Uses context, e.g., can distinguish the noun "bit" from the verb form "bit" (of "bite") | About 25× slower on our data (minutes instead of seconds); wrong POS tags lead to wrong lemmas |
| **Transformer pipeline** (`en_core_web_trf`) | Same rules, but POS tags are predicted by a large transformer model | Most accurate POS tags | Much slower again, ideally needs a GPU — overkill for a bag-of-words model |

A related, cruder technique is *stemming* (e.g., the Porter stemmer in the `nltk` package), which simply chops off word endings ("balanced" → "balanc"). Stems are often not real words, and spaCy does not offer stemming.

When we compared the lookup lemmatizer with `en_core_web_sm` on all 130k wine reviews, about 4% of the tokens received a different lemma. Some typical differences:

| Word | Statistical pipeline | Lookup | Comment |
|---|---|---|---|
| bit | bit | bite | lookup ignores that "a bit" is a noun |
| clove | clove | cleave | — " — |
| cola | cola | colon | — " — |
| balanced | balanced | balance | adjective vs. verb: debatable |
| zest | z | zest | statistical pipeline wrongly tags "zest" as an adjective |
| candied | candy | candied | statistical pipeline treats "candied" as a verb |

Do these differences matter for our task? Hardly: as you will see later, the classifier's accuracy and F1-score are practically identical with both lemmatizers. For bag-of-words models, a fast lemmatizer is usually good enough. For tasks that depend on the exact meaning of individual words, the slower context-aware options are worth it.

# Vectorize documents

Vectorization is the process of turning a collection of text documents into numerical feature vectors.

We will use the **Bag of Words (BoW)** model for vectorization. In the BoW model, a corpus of documents is represented by a matrix with one row per document and one column per word occurring in the corpus. The cell values will either be simple frequency counts (How often does a word appear in a document?), or the term frequency (tf) times the inverse document frequency (idf) of a term. The idea of tf-idf is to scale down the impact of words that occur very frequently in a given corpus and that are therefore less informative than features that occur only in a small fraction of the corpus. Note that the BoW model completely ignores information about the position and sequences of the words in the document.

In `sklearn`, the [`CountVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html#sklearn.feature_extraction.text.CountVectorizer) creates a term-document matrix with (normalized) term frequencies and the [`TfIdfVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html#sklearn.feature_extraction.text.TfidfVectorizer) creates a term-document matrix with tf-idf weighting.

In [ ]:
count_vect = CountVectorizer(min_df=10)

Apply the CountVectorizer object to the review texts of the training set.

In [ ]:
X_training = count_vect.fit_transform(training["description_prep"])

Display an extract of the generated term-document matrix

In [ ]:
X_training.shape

In [ ]:
X_training[0:20,0:20].todense()

Store the labels that we want to predict in a separate variable.

In [ ]:
y_training = training["verygood"]
y_training.describe()

# Train classifier on training set

Fit a logistic regression classification with the term-document matrix as the features and the wine quality (i.e., `verygood` variable) as the label.

In [ ]:
clf = LogisticRegression(max_iter=1000).fit(X_training, y_training)

Test whether classifier is working by predicting the quality of a short fake review. We apply the same NLP preprocessing steps and reuse the `count_vect` object to generate features in the same way as we did for the training set.

In [ ]:
doc_new = {'description': ['This is a spectacular, magnificent, and majestic wine. Awesome!']}
doc_new_df = pd.DataFrame.from_dict(doc_new)

In [ ]:
doc_new_df_prep = spacy_prep_df(doc_new_df)
doc_new_df_prep

In [ ]:
X_new = count_vect.transform(doc_new_df_prep["description_prep"])

In [ ]:
predicted = clf.predict(X_new)
predicted

Instead of predicting binary labels, we can also predict probabilities of the classes.

In [ ]:
predicted_prob = clf.predict_proba(X_new)
print(clf.classes_)
print(predicted_prob)

# Evaluate accuracy on validation set

Before trying to predict the labels for the official test set, we evaluate the predictive accuracy of our model on the validation set. Again, we apply the same NLP preprocessing steps, reuse the `count_vect` object, and store `X` and `y` in separate data structures.

In [ ]:
validation = spacy_prep_df(validation)

In [ ]:
X_validation = count_vect.transform(validation["description_prep"])
y_validation = validation["verygood"]

Call the predict function of our model with the validation data and calculate precision, recall and F1-score.

The Kaggle competition ranks submissions by the **macro-averaged F1-score**, i.e., the unweighted mean of the F1-scores of both classes (shown in the row `macro avg` of the report). Unlike accuracy, it cannot be gamed by always predicting the majority class: predicting "not very good" for every wine gives an accuracy of about 0.72, but a macro F1-score of only about 0.42.

In [ ]:
predictions_validation = clf.predict(X_validation)
print(metrics.classification_report(y_validation, predictions_validation))
print("Macro F1-score:", metrics.f1_score(y_validation, predictions_validation, average="macro"))

# Interpret model

Logistic regression is typically not the most accurate classification model, but one big advantage is that it can be interpreted by looking at the coefficients of the input features.

In [ ]:
coeffs = clf.coef_[0].tolist()
words = count_vect.get_feature_names_out()
words_with_coeffs = pd.DataFrame(coeffs, words, columns=["coeff"])

These are the words with the most *negative* impact.    

In [ ]:
words_with_coeffs.sort_values("coeff", ascending=True).head(10)

And these are the words with the most *positive* impact.

In [ ]:
words_with_coeffs.sort_values("coeff", ascending=False).head(10)

# Make predictions on test set

Time to compete! Join the course's [Kaggle competition](https://www.kaggle.com/t/208059e8f1434629a60386479a523af0), where you can upload your predictions for the test set and see how your model ranks against your fellow students.

Load the official test set. It contains the same columns as the training data, except for the `points`: these are only known to Kaggle, which uses them to score your predictions. Preprocess and vectorize the review texts of the test set.

In [ ]:
test = pd.read_csv("https://raw.githubusercontent.com/olivermueller/amlta-2026/main/Session_01/wine_reviews_test.csv")
test = spacy_prep_df(test)

In [ ]:
X_test = count_vect.transform(test["description_prep"])
predictions_test = clf.predict(X_test)

Create a dataframe with the ids of the test reviews and our predictions and save it as a CSV file, which we can upload to the [Kaggle competition](https://www.kaggle.com/t/208059e8f1434629a60386479a523af0) (go to *Submit Prediction*). In Colab, you can download the file via the *Files* panel on the left.

In [ ]:
my_submission = pd.DataFrame({'id': test["id"],
                              'verygood': predictions_test})

In [ ]:
my_submission.head()

In [ ]:
my_submission.to_csv("my_submission.csv", index=False)

Do feature engineering and try different learning algorithms to improve your model's performance. For example, you can try different preprocessing steps, such as removing stop words, using n-grams, or applying different vectorization techniques. In addition, try different classification algorithms, such as random forests or neural networks. You can also experiment with hyperparameter tuning to find the best settings for your model.